## Binary Customer Churn Prediction

A marketing agency wants to predict which customers will churn (stop buying their service) so they can assign at-risk customers an account manager.

**Features:**
- **Name**: Name of the latest contact at Company
- **Age**: Customer Age
- **Total_Purchase**: Total Ads Purchased
- **Account_Manager**: Binary 0=No manager, 1=Account manager assigned
- **Years**: Total Years as a customer
- **Num_Sites**: Number of websites that use the service
- **Onboard_date**: Date that the latest contact was onboarded
- **Location**: Client HQ Address
- **Company**: Name of Client Company
- **Churn**: Target variable (1=churned, 0=stayed)

### 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    precision_recall_curve
)
%matplotlib inline

### 2. Load and Explore Data

In [ ]:
df = pd.read_csv('customer_churn.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### 3. Data Preprocessing

In [ ]:
# Drop non-numeric columns that are not useful for the model
df.drop(['Names', 'Location', 'Company', 'Onboard_date'], axis=1, inplace=True)
df.dtypes

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Check class distribution
print('Churn distribution:')
print(df['Churn'].value_counts())
print(f'\nChurn rate: {df["Churn"].mean()*100:.2f}%')

### 4. Define Model Functions

In [ ]:
# Sigmoid function - squashes any real number into (0, 1)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# Cost function - binary cross-entropy loss and gradient
def costFunction(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    error = (y * np.log(h) + (1 - y) * np.log(1 - h))
    cost = -1/m * sum(error)
    grad = 1/m * X.T.dot(h - y)
    return cost, grad

In [ ]:
# Gradient descent optimizer
def gradientDescent(X, y, theta, alpha, iterations):
    cost_history = np.zeros(iterations)
    for i in range(iterations):
        cost, grad = costFunction(X, y, theta)
        theta -= alpha * grad
        cost_history[i] = cost
    return theta, cost_history

In [ ]:
# Predict function - returns probabilities
def predict(X, theta):
    return sigmoid(X.dot(theta))

In [ ]:
# Accuracy function
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

### 5. Train/Test Split and Feature Scaling

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('Churn', axis=1), df['Churn'],
    test_size=0.2, random_state=42
)

In [ ]:
# Standardize features (mean=0, std=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Add intercept column (column of 1s for the bias term)
X_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test = np.c_[np.ones((X_test.shape[0], 1)), X_test]

### 6. Train the Model

In [ ]:
# Run gradient descent with learning rate=0.01 for 1000 iterations
theta, cost_history = gradientDescent(
    X_train, y_train,
    np.zeros(X_train.shape[1]),
    alpha=0.01, iterations=1000
)

In [ ]:
# Plot cost convergence
plt.figure(figsize=(8, 5))
plt.plot(range(1000), cost_history)
plt.xlabel('Iterations')
plt.ylabel('Cost')
plt.title('Cost History - Gradient Descent Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7. Predictions on Test Set

In [ ]:
# Get probabilities and convert to binary predictions (threshold=0.5)
y_probs = predict(X_test, theta)
y_pred = [1 if i > 0.5 else 0 for i in y_probs]

### 8. Model Evaluation

In [ ]:
# Key metrics
print('MODEL EVALUATION METRICS')
print('=' * 40)
print(f'Accuracy:  {accuracy(y_test, y_pred)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred)*100:.2f}%')

In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=['No Churn (0)', 'Churn (1)']))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn (0)', 'Churn (1)'],
            yticklabels=['No Churn (0)', 'Churn (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

### 9. Precision vs Recall Curve

In [ ]:
# Precision-Recall curve at different thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

plt.figure(figsize=(8, 5))
plt.plot(recalls, precisions, color='blue', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision vs Recall Curve')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.grid(True, alpha=0.3)

# Mark the default 0.5 threshold point
default_precision = precision_score(y_test, y_pred)
default_recall = recall_score(y_test, y_pred)
plt.scatter(default_recall, default_precision, color='red', s=100, zorder=5,
            label=f'Threshold=0.5 (P={default_precision:.2f}, R={default_recall:.2f})')
plt.legend()
plt.tight_layout()
plt.show()

### 10. Predict on New Customers

In [39]:
# Load new customers
df2 = pd.read_csv('new_customers_1.csv')
df2.head()

,Names,Age,Total_Purchase,Account_Manager,Years,Num_Sites,Onboard_date,Location,Company
0,Andrew Mccall,37.0,9935.53,1,7.71,8.0,2011-08-29 18:37:54,38612 Johnny Stravenue Nataliebury WI 15717-8316,King Ltd
1,Michele Wright,23.0,7526.94,1,9.28,15.0,2013-07-22 18:19:54,"21083 Nicole Junction Suite 332, Youngport ME ...",Cannon-Benson
2,Jeremy Chang,65.0,100.00,1,1.00,15.0,2006-12-11 07:48:13,085 Austin Views Lake Julialand WY 63726-4298,Barron-Robertson
3,Megan Ferguson,32.0,6487.50,0,9.40,14.0,2016-10-28 05:32:13,922 Wright Branch North Cynthialand NC 64721,Sexton-Golden
4,Taylor Young,32.0,13147.71,1,10.00,8.0,2012-03-20 00:36:46,Unit 0789 Box 0734 DPO AP 39702,Wood LLC


In [40]:
# Store names for later, then drop non-numeric columns
customer_names = df2['Names'].tolist()
df2.drop(['Names', 'Location', 'Company', 'Onboard_date'], axis=1, inplace=True)

# Apply same scaling and add intercept
X_new = scaler.transform(df2)
X_new = np.c_[np.ones((X_new.shape[0], 1)), X_new]

# Predict
y_pred_new_probs = predict(X_new, theta)
y_pred_new = [1 if i > 0.5 else 0 for i in y_pred_new_probs]

# Add predictions to dataframe
df2.insert(0, 'Names', customer_names)
df2['Churn_Probability'] = y_pred_new_probs.round(4)
df2['Churn_Prediction'] = y_pred_new
df2['Action'] = df2['Churn_Prediction'].map({1: 'Assign Account Manager', 0: 'No Action Needed'})
df2

,Names,Age,Total_Purchase,Account_Manager,Years,Num_Sites,Churn_Probability,Churn_Prediction,Action
0,Andrew Mccall,37.0,9935.53,1,7.71,8.0,0.2410,0,No Action Needed
1,Michele Wright,23.0,7526.94,1,9.28,15.0,0.9281,1,Assign Account Manager
2,Jeremy Chang,65.0,100.00,1,1.00,15.0,0.7777,1,Assign Account Manager
3,Megan Ferguson,32.0,6487.50,0,9.40,14.0,0.8793,1,Assign Account Manager
4,Taylor Young,32.0,13147.71,1,10.00,8.0,0.3463,0,No Action Needed
5,Jessica Drake,22.0,8445.26,1,3.46,14.0,0.6526,1,Assign Account Manager
